In [ ]:
# ------------------------------- IMPORTS -------------------------------
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, auc)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import joblib

# !pip install tabpfn imbalanced-learn category_encoders torch

try:
    from tabpfn import TabPFNClassifier
except Exception as e:
    raise ImportError("No se encontró TabPFN. En Colab ejecute: !pip install tabpfn --quiet y vuelva a correr.\n" + str(e))

from imblearn.over_sampling import ADASYN
import torch

In [ ]:
# ------------------------------- CONFIG -------------------------------
RANDOM_STATE = 42
N_SPLITS = 5
MAX_TABPFN = 1000
RESULTS_DIR = os.path.join("outputs", "TABPFN")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------- FUNCIONES -------------------------------

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def load_data(path="data/Cleaned.csv", target_col="NoShow"):
    df = pd.read_csv(path)
    if target_col not in df.columns:
        raise ValueError(f"Target {target_col} no existe.")
    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].copy()
    return df, X, y

def preprocess(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    X = X.copy()
    for c in num_cols:
        X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].fillna("__MISSING__")
    enc_cat = None
    if len(cat_cols) > 0:
        enc_cat = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[cat_cols] = enc_cat.fit_transform(X[cat_cols])
    scaler = None
    if len(num_cols) > 0:
        scaler = StandardScaler()
        X[num_cols] = scaler.fit_transform(X[num_cols])
    feature_names = num_cols + cat_cols
    return X[feature_names].to_numpy(dtype=float)

def evaluate_metrics(y_true, y_pred, y_proba):
    res = {}
    res['accuracy'] = accuracy_score(y_true, y_pred)
    res['precision'] = precision_score(y_true, y_pred, zero_division=0)
    res['recall'] = recall_score(y_true, y_pred, zero_division=0)
    res['f1'] = f1_score(y_true, y_pred, zero_division=0)
    try:
        res['roc_auc'] = roc_auc_score(y_true, y_proba[:,1])
    except:
        res['roc_auc'] = np.nan
    try:
        res['average_precision'] = average_precision_score(y_true, y_proba[:,1])
    except:
        res['average_precision'] = np.nan
    return res

def split_into_chunks(X, y, max_samples=1000):
    n = X.shape[0]
    for i in range(0, n, max_samples):
        yield X[i:i+max_samples], y[i:i+max_samples]

In [ ]:
# ------------------------------- PIPELINE -------------------------------

def run_pipeline(csv_path="data/Cleaned.csv", target_col="NoShow"):
    set_seed(RANDOM_STATE)
    df, X_raw, y_raw = load_data(csv_path, target_col)
    X_all = preprocess(X_raw)
    y_all = y_raw.to_numpy()

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    all_metrics = []
    all_trues, all_preds, all_probas = [], [], []

    chunk_id = 0
    for X_chunk, y_chunk in split_into_chunks(X_all, y_all, MAX_TABPFN):
        print(f"Processing chunk {chunk_id+1} with {X_chunk.shape[0]} samples")
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_chunk, y_chunk)):
            X_train, X_test = X_chunk[train_idx], X_chunk[test_idx]
            y_train, y_test = y_chunk[train_idx], y_chunk[test_idx]

            # Balancear con ADASYN
            ada = ADASYN(random_state=RANDOM_STATE)
            X_train_bal, y_train_bal = ada.fit_resample(X_train, y_train)

            # Forzar a máximo 1000 registros después de ADASYN
            if X_train_bal.shape[0] > MAX_TABPFN:
                idx = np.random.choice(X_train_bal.shape[0], MAX_TABPFN, replace=False)
                X_train_bal = X_train_bal[idx]
                y_train_bal = y_train_bal[idx]

            clf = TabPFNClassifier(device="cuda")
            clf.fit(X_train_bal.astype(float), y_train_bal.astype(int))

            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)

            metrics = evaluate_metrics(y_test, y_pred, y_proba)
            metrics['chunk'] = chunk_id
            metrics['fold'] = fold_idx
            all_metrics.append(metrics)

            all_trues.extend(y_test.tolist())
            all_preds.extend(y_pred.tolist())
            all_probas.extend(y_proba[:,1].tolist())

            # t-SNE solo para un fold por chunk
            if fold_idx == 0:
                tsne = TSNE(n_components=2, random_state=RANDOM_STATE)
                emb = tsne.fit_transform(X_test[:min(500, len(X_test))])
                plt.figure(figsize=(6,5))
                plt.scatter(emb[:,0], emb[:,1], c=y_test[:len(emb)], cmap='Spectral', s=20)
                plt.title(f"t-SNE chunk{chunk_id} fold{fold_idx}")
                plt.savefig(os.path.join(RESULTS_DIR, f"tsne_chunk{chunk_id}_fold{fold_idx}.png"))
                plt.close()
        chunk_id += 1

    # Agregar métricas globales
    y_trues = np.array(all_trues)
    y_preds = np.array(all_preds)
    y_probas = np.array(all_probas)

    final_metrics = evaluate_metrics(y_trues, y_preds, np.vstack([1-y_probas, y_probas]).T)
    print("\nFinal aggregated metrics:")
    for k,v in final_metrics.items():
        print(f"{k}: {v:.4f}")

    # Guardar métricas por fold
    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_csv(os.path.join(RESULTS_DIR, "fold_metrics.csv"), index=False)

    # Guardar media y desviación estándar
    agg = metrics_df.drop(columns=["chunk", "fold"]).agg(["mean", "std"])
    agg.to_csv(os.path.join(RESULTS_DIR, "metrics_summary.csv"))

    # ROC curve
    fpr, tpr, _ = roc_curve(y_trues, y_probas)
    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC AUC={auc(fpr,tpr):.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.savefig(os.path.join(RESULTS_DIR, "roc_curve.png"))
    plt.close()

    # Confusion matrix
    cm = confusion_matrix(y_trues, y_preds)
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion matrix")
    plt.colorbar()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i,j]), ha='center', va='center')
    plt.xlabel("Prediction")
    plt.ylabel("Real")
    plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"))
    plt.close()

    pd.DataFrame([final_metrics]).to_csv(os.path.join(RESULTS_DIR, "final_metrics.csv"), index=False)

    return final_metrics

# ------------------------------- MAIN -------------------------------
if __name__ == "__main__":
    results = run_pipeline("data/Cleaned.csv", "NoShow")
    print(results)


Processing chunk 1 with 1000 samples
Processing chunk 2 with 1000 samples
Processing chunk 3 with 1000 samples
Processing chunk 4 with 1000 samples
Processing chunk 5 with 1000 samples
Processing chunk 6 with 1000 samples
Processing chunk 7 with 1000 samples
Processing chunk 8 with 1000 samples
Processing chunk 9 with 1000 samples
Processing chunk 10 with 1000 samples
Processing chunk 11 with 1000 samples
Processing chunk 12 with 1000 samples
Processing chunk 13 with 1000 samples
Processing chunk 14 with 1000 samples
Processing chunk 15 with 1000 samples
Processing chunk 16 with 1000 samples
Processing chunk 17 with 1000 samples
Processing chunk 18 with 1000 samples
Processing chunk 19 with 1000 samples
Processing chunk 20 with 1000 samples
Processing chunk 21 with 1000 samples
Processing chunk 22 with 1000 samples
Processing chunk 23 with 1000 samples
Processing chunk 24 with 1000 samples
Processing chunk 25 with 1000 samples
Processing chunk 26 with 1000 samples
Processing chunk 27 w